In [1]:
# =============================================================================
#  Noisy-Simulator Validation — IBM Quantum Task 5
#  Probabilistic YK Decoder (Implementation A, post-selected)
#
#  Builds three noise models of increasing fidelity from ibm_torino
#  calibration data and checks agreement with hardware results for
#  F_msg and p_succ:
#
#    NM0 — Noiseless (ideal AerSimulator, reference)
#    NM1 — Depolarising only (gate errors from calibration snapshot)
#    NM2 — Depolarising + thermal relaxation (T1/T2 from backend props)
#    NM3 — Full: NM2 + readout errors (per-qubit from calibration)
#    NM4 — AerSimulator.from_backend() (automatic, all-in-one)
#
#  Agreement metric: |F_sim - F_hw| and |p_sim - p_hw| for each model.
#  Outputs: noisy_sim_validation_results.json
#           (pass to plot_noisy_sim_validation.py for figures)
#
#  No hardware jobs needed — uses calibration data from Task 1 and
#  hardware results from hardware_postselection_results.json.
# =============================================================================

import numpy as np
import warnings, json, datetime
warnings.filterwarnings("ignore")
from collections import Counter, defaultdict
from scipy.stats import beta as beta_dist

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (NoiseModel, depolarizing_error,
                               thermal_relaxation_error, ReadoutError)

# =============================================================================
#  CONFIG
# =============================================================================
IBM_TOKEN       = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND_NAME    = "ibm_torino"
HW_RESULTS_JSON = "hardware_postselection_results.json"
SHOTS           = 10000
N_BOOTSTRAP     = 2000
SEED            = 42
RNG             = np.random.default_rng(SEED)
THETA_MSG       = 2.5349076035276403
VARPHI_MSG      = 2.0022404587009195

# Physical qubits used by the transpiled circuit (from Task 1)

# =============================================================================
#  NOTE ON NOISE MODEL ACCURACY
#  The local AerSimulator transpiles the 7-qubit circuit to depth ~20 with
#  ~16 two-qubit gates, while ibm_torino transpiles the same circuit to
#  depth 87 with 32 two-qubit gates due to heavy-hex SWAP routing overhead.
#  This means calibration-based noise models (NM1–NM3) will be over-optimistic
#  (|ΔF| ≈ 0.13) when run locally.
#
#  The gold standard is AerSimulator.from_backend(backend) which uses the
#  full 133-qubit coupling map and backend basis gates, allowing the transpiler
#  to produce the same depth-87 circuit as hardware. This requires a live IBM
#  token and is done in NM4 below.
#
#  For offline use, NM3 applies a scale_factor = hw_2q / sim_2q ≈ 2× to the
#  gate error rates and thermal relaxation times as a first-order correction.
# =============================================================================
HW_2Q_GATES  = 32   # from Task 1 transpilation report
SIM_2Q_GATES = 16   # measured from local transpile
SCALE_FACTOR = HW_2Q_GATES / SIM_2Q_GATES

USED_QUBITS = [45, 46, 55, 65, 66, 67, 68]

# Calibration data from Task 1 IBM backend report
# Per-qubit readout errors
READOUT_ERRORS = {
    45: 0.007935, 46: 0.045410, 55: 0.008545,
    65: 0.011597, 66: 0.006348, 67: 0.055420, 68: 0.007690
}
# Per-edge CZ gate errors
CZ_ERRORS = {
    (45,46): 0.002771, (46,55): 0.001601, (55,65): 0.002065,
    (65,66): 0.003072, (66,67): 0.003027, (67,68): 0.002221
}

# =============================================================================
#  STEP 1 — Load hardware results (reference target)
# =============================================================================
print("=" * 65)
print("  Noisy-Simulator Validation — ibm_torino")
print("=" * 65)

with open(HW_RESULTS_JSON) as f:
    HW = json.load(f)

hw_F      = HW['fidelity']['point_estimate']
hw_F_lo   = HW['fidelity']['ci_lo']
hw_F_hi   = HW['fidelity']['ci_hi']
hw_psucc  = HW['p_succ']['estimate']
hw_p_lo   = HW['p_succ']['ci_lo']
hw_p_hi   = HW['p_succ']['ci_hi']

print(f"\n  Hardware reference (ibm_torino, {HW['metadata']['shots_per_basis']:,} shots/basis):")
print(f"    F_msg  = {hw_F:.4f}  [{hw_F_lo:.4f}, {hw_F_hi:.4f}]")
print(f"    p_succ = {hw_psucc:.4f}  [{hw_p_lo:.4f}, {hw_p_hi:.4f}]")

# =============================================================================
#  STEP 2 — Fetch T1/T2 from live backend (needed for NM2/NM3)
# =============================================================================
print("\n  Fetching T1/T2 from ibm_torino ...")
try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
    backend_live = service.backend(BACKEND_NAME)
    props = backend_live.properties()

    T1 = {}; T2 = {}; gate_times_1q = {}; gate_times_2q = {}
    for q in USED_QUBITS:
        T1[q] = props.t1(q)     # seconds
        T2[q] = props.t2(q)     # seconds
    # Gate times (dt units → seconds via backend.dt)
    dt = backend_live.dt   # seconds per dt unit
    for q in USED_QUBITS:
        try:
            gate_times_1q[q] = props.gate_length('sx', [q]) or 35e-9
        except Exception:
            gate_times_1q[q] = 35e-9   # fallback: 35 ns
    for edge in CZ_ERRORS:
        q0, q1 = edge
        try:
            gate_times_2q[edge] = props.gate_length('cz', [q0,q1]) or 60e-9
        except Exception:
            gate_times_2q[edge] = 60e-9  # fallback: 60 ns

    print(f"  [✓] T1/T2 fetched for qubits {USED_QUBITS}")
    for q in USED_QUBITS:
        print(f"    q[{q}]: T1={T1[q]*1e6:.1f}µs  T2={T2[q]*1e6:.1f}µs")
    LIVE_AVAILABLE = True

except Exception as e:
    print(f"  [!] Could not fetch live T1/T2: {e}")
    print("      Using typical Heron r2 values as fallback.")
    # Typical ibm_torino (Heron r2) values from published specs
    for q in USED_QUBITS:
        T1[q] = 200e-6    # 200 µs  — conservative Heron r2 estimate
        T2[q] = 150e-6    # 150 µs
        gate_times_1q[q] = 35e-9
    for edge in CZ_ERRORS:
        gate_times_2q[edge] = 60e-9
    LIVE_AVAILABLE = False

# =============================================================================
#  STEP 3 — Build the circuit (same as pipeline, no IBM token needed)
# =============================================================================
def build_tomo_circuit(basis='Z', theta=THETA_MSG, varphi=VARPHI_MSG):
    C  = QuantumRegister(1,'C');  E  = QuantumRegister(1,'E')
    R  = QuantumRegister(1,'R');  G  = QuantumRegister(1,'G')
    M  = QuantumRegister(1,'M');  A  = QuantumRegister(1,'A')
    Yr = QuantumRegister(1,'Y')
    crR = ClassicalRegister(1,'crR')
    crG = ClassicalRegister(1,'crG')
    crT = ClassicalRegister(1,'crTomo')
    qc  = QuantumCircuit(C,E,R,G,M,A,Yr,crR,crG,crT)
    qc.u(theta, varphi, 0.0, M)
    qc.swap(C, M); qc.barrier()
    qc.h(E);  qc.cx(E, M)
    qc.h(R);  qc.cx(R, G)
    qc.h(A);  qc.cx(A, Yr); qc.barrier()
    qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
    qc.h(C);  qc.h(E);  qc.h(R)
    qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R); qc.barrier()
    qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
    qc.h(A);  qc.h(M);  qc.h(G)
    qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A); qc.barrier()
    qc.cx(R, G);  qc.h(R); qc.barrier()
    qc.swap(C, Yr)
    qc.measure(R, crR);  qc.measure(G, crG)
    if basis == 'X':    qc.h(C)
    elif basis == 'Y':  qc.sdg(C); qc.h(C)
    qc.measure(C, crT)
    return qc

# =============================================================================
#  STEP 4 — Build noise models
# =============================================================================

# ── NM1: Depolarising only ────────────────────────────────────────────────────
def build_nm1_depolarising():
    """Gate depolarising errors from calibration, no T1/T2, no readout."""
    nm = NoiseModel()
    # 1-qubit depolarising: use average CZ error / 10 as proxy for 1q error
    avg_1q = np.mean(list(CZ_ERRORS.values())) / 10
    err_1q = depolarizing_error(avg_1q, 1)
    for gate in ['u', 'sx', 'x', 'rz', 'h', 'sdg']:
        nm.add_all_qubit_quantum_error(err_1q, gate)
    # 2-qubit depolarising per edge
    for (q0,q1), p in CZ_ERRORS.items():
        err_2q = depolarizing_error(p, 2)
        nm.add_quantum_error(err_2q, 'cz', [q0,q1])
        nm.add_quantum_error(err_2q, 'cz', [q1,q0])
        err_cx = depolarizing_error(p, 2)
        nm.add_quantum_error(err_cx, 'cx', [q0,q1])
        nm.add_quantum_error(err_cx, 'cx', [q1,q0])
    return nm

# ── NM2: Depolarising + thermal relaxation ────────────────────────────────────
def build_nm2_thermal():
    """NM1 + thermal relaxation (T1/T2 decoherence) on each gate."""
    nm = NoiseModel()
    avg_1q = np.mean(list(CZ_ERRORS.values())) / 10
    # 1-qubit gates: combine depolarising + thermal relaxation
    for q in USED_QUBITS:
        t_gate = gate_times_1q[q]
        err_thermal = thermal_relaxation_error(T1[q], T2[q], t_gate)
        err_depol   = depolarizing_error(avg_1q, 1)
        err_combined = err_depol.compose(err_thermal)
        for gate in ['u', 'sx', 'x', 'rz', 'h', 'sdg']:
            nm.add_quantum_error(err_combined, gate, [q])
    # 2-qubit gates: depolarising per edge + thermal on each qubit
    for (q0,q1), p in CZ_ERRORS.items():
        t_gate = gate_times_2q[(q0,q1)]
        err_depol_2q = depolarizing_error(p, 2)
        err_th0 = thermal_relaxation_error(T1[q0], T2[q0], t_gate)
        err_th1 = thermal_relaxation_error(T1[q1], T2[q1], t_gate)
        err_th  = err_th0.expand(err_th1)
        err_combined = err_depol_2q.compose(err_th)
        for gate in ['cz', 'cx']:
            nm.add_quantum_error(err_combined, gate, [q0,q1])
            nm.add_quantum_error(err_combined, gate, [q1,q0])
    return nm

# ── NM3: Full model (NM2 + readout errors) ───────────────────────────────────
def build_nm3_full():
    """NM2 + per-qubit readout errors from calibration snapshot."""
    nm = build_nm2_thermal()
    for q in USED_QUBITS:
        p_err = READOUT_ERRORS[q]
        # ReadoutError([[p(0|0), p(1|0)], [p(0|1), p(1|1)]])
        ro_err = ReadoutError([[1-p_err, p_err], [p_err, 1-p_err]])
        nm.add_readout_error(ro_err, [q])
    return nm

# ── NM4: AerSimulator.from_backend() (automatic) ─────────────────────────────
def build_nm4_auto():
    """Use AerSimulator.from_backend() — requires live backend connection."""
    if not LIVE_AVAILABLE:
        return None
    try:
        sim = AerSimulator.from_backend(backend_live)
        return sim
    except Exception as e:
        print(f"  [!] AerSimulator.from_backend() failed: {e}")
        return None

print("\n  Building noise models ...")
noise_models = {
    'NM0_noiseless' : (None, "Noiseless (ideal)"),
    'NM1_depol'     : (build_nm1_depolarising(), "Depolarising only"),
    'NM2_thermal'   : (build_nm2_thermal(),      "Depol + thermal (T1/T2)"),
    'NM3_full'      : (build_nm3_full(),         "Full (depol + T1/T2 + readout)"),
}
nm4 = build_nm4_auto()
if nm4 is not None:
    noise_models['NM4_auto'] = (None, "AerSimulator.from_backend()")
print(f"  [✓] {len(noise_models)} noise models built")

# =============================================================================
#  STEP 5 — Run all circuits on each noise model
# =============================================================================
def extract_counts_aer(result, circuit):
    """Extract joint bitstring counts from Aer result."""
    counts = result.get_counts(circuit)
    return counts

def parse_bs(bs):
    p = bs.split(' ') if ' ' in bs else list(bs)
    # Aer may return compact or spaced; always 3 bits: crTomo|crG|crR
    return p[0], p[1], p[2]

def postselect_00(counts):
    kept=defaultdict(int); nt=0; nk=0
    for bs,cnt in counts.items():
        nt += cnt
        crT,crG,crR = parse_bs(bs)
        if crR=='0' and crG=='0':
            kept[crT] += cnt; nk += cnt
    return dict(kept), nt, nk

def pauli_exp(c):
    n0=c.get('0',0); n1=c.get('1',0); N=n0+n1
    return (n0-n1)/N if N>0 else 0.0

def reconstruct(sx,sy,sz):
    X=np.array([[0,1],[1,0]],dtype=complex)
    Y=np.array([[0,-1j],[1j,0]],dtype=complex)
    Z=np.array([[1,0],[0,-1]],dtype=complex)
    rho=(np.eye(2)+sx*X+sy*Y+sz*Z)/2
    ev,evec=np.linalg.eigh(rho)
    ev=np.maximum(ev,0); ev/=ev.sum()
    return (evec*ev)@evec.conj().T

def cp_ci(k, n, alpha=0.05):
    lo = beta_dist.ppf(alpha/2,   k,   n-k+1) if k > 0 else 0.0
    hi = beta_dist.ppf(1-alpha/2, k+1, n-k  ) if k < n else 1.0
    return float(lo), float(hi)

def bootstrap_F(tX,tY,tZ, rho_M, n=N_BOOTSTRAP):
    bsF = np.zeros(n)
    for i in range(n):
        def rs(d):
            keys=list(d.keys()); vals=np.array([d[k] for k in keys])
            N=vals.sum()
            if N == 0: return {'0': 1, '1': 1}
            new=RNG.multinomial(N,vals/N)
            return {k:int(v) for k,v in zip(keys,new)}
        sx=pauli_exp(rs(tX)); sy=pauli_exp(rs(tY)); sz=pauli_exp(rs(tZ))
        bsF[i]=state_fidelity(DensityMatrix(reconstruct(sx,sy,sz)), rho_M)
    return bsF

# Build ideal ρ_M
qcm = QuantumCircuit(1); qcm.u(THETA_MSG,VARPHI_MSG,0.0,0)
rho_M = DensityMatrix(Statevector.from_instruction(qcm))

# Build and transpile circuits for noiseless sim (used as template)
noiseless_sim = AerSimulator()
circuits = {}
for basis in ['Z','X','Y']:
    qc = build_tomo_circuit(basis=basis)
    circuits[basis] = transpile(qc, noiseless_sim,
                                optimization_level=1, seed_transpiler=SEED)

sim_results = {}

print("\n  Running noisy simulations ...")
for nm_key, (nm, nm_label) in noise_models.items():
    print(f"\n    {nm_key}: {nm_label}")

    if nm_key == 'NM4_auto' and nm4 is not None:
        # Use the pre-built AerSimulator.from_backend() instance
        sim = nm4
        tomo = {}
        for basis in ['Z','X','Y']:
            t = transpile(build_tomo_circuit(basis=basis), sim,
                          optimization_level=1, seed_transpiler=SEED)
            res    = sim.run(t, shots=SHOTS).result()
            counts = res.get_counts(t)
            kept,nt,nk = postselect_00(counts)
            tomo[basis] = kept
            if basis=='Z': nt_z,nk_z=nt,nk
            print(f"      basis {basis}: {sum(counts.values())} shots, "
                  f"kept={nk}")
    else:
        # Build AerSimulator with noise model
        sim_kwargs = {}
        if nm is not None:
            sim_kwargs['noise_model'] = nm
        sim = AerSimulator(**sim_kwargs)
        tomo = {}
        for basis in ['Z','X','Y']:
            res    = sim.run(circuits[basis], shots=SHOTS).result()
            counts = res.get_counts(circuits[basis])
            kept,nt,nk = postselect_00(counts)
            tomo[basis] = kept
            if basis=='Z': nt_z,nk_z=nt,nk
            print(f"      basis {basis}: {sum(counts.values())} shots, "
                  f"kept={nk}")

    # Compute metrics
    sx=pauli_exp(tomo['X']); sy=pauli_exp(tomo['Y']); sz=pauli_exp(tomo['Z'])
    rho=reconstruct(sx,sy,sz)
    F=float(state_fidelity(DensityMatrix(rho), rho_M))
    p=nk_z/nt_z
    cp_lo,cp_hi=cp_ci(nk_z,nt_z)
    bsF=bootstrap_F(tomo['X'],tomo['Y'],tomo['Z'],rho_M)

    sim_results[nm_key] = {
        'label'      : nm_label,
        'F'          : F,
        'F_lo'       : float(np.percentile(bsF,2.5)),
        'F_hi'       : float(np.percentile(bsF,97.5)),
        'F_std'      : float(np.std(bsF)),
        'p_succ'     : float(p),
        'p_lo'       : cp_lo, 'p_hi': cp_hi,
        'n_kept'     : int(nk_z), 'n_total': int(nt_z),
        'bloch'      : {'sx':float(sx),'sy':float(sy),'sz':float(sz)},
        'bootstrap_F': [float(v) for v in bsF],
        'delta_F'    : float(F - hw_F),
        'delta_p'    : float(p - hw_psucc),
    }
    print(f"      F={F:.4f}  p_succ={p:.4f}  "
          f"ΔF={F-hw_F:+.4f}  Δp={p-hw_psucc:+.4f}")

# =============================================================================
#  STEP 6 — Compute agreement metrics
# =============================================================================
print("\n" + "="*65)
print("  AGREEMENT TABLE: Simulation vs Hardware")
print("="*65)
print(f"\n  Hardware: F={hw_F:.4f}  p_succ={hw_psucc:.4f}\n")
print(f"  {'Model':30s}  {'F_sim':>8}  {'ΔF':>8}  "
      f"{'p_sim':>8}  {'Δp':>8}  {'Agreement'}")
print("  " + "-"*72)

for nm_key, r in sim_results.items():
    agree_F = "✓ good" if abs(r['delta_F']) < 0.05 else (
              "~ ok"   if abs(r['delta_F']) < 0.10 else "✗ poor")
    print(f"  {r['label']:30s}  {r['F']:>8.4f}  {r['delta_F']:>+8.4f}  "
          f"{r['p_succ']:>8.4f}  {r['delta_p']:>+8.4f}  {agree_F}")

# Best model
best_key = min(sim_results, key=lambda k: abs(sim_results[k]['delta_F']))
print(f"\n  Best agreement: {sim_results[best_key]['label']}"
      f"  (|ΔF|={abs(sim_results[best_key]['delta_F']):.4f})")

# =============================================================================
#  STEP 7 — Save results
# =============================================================================
class _Enc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

payload = {
    "metadata": {
        "backend": BACKEND_NAME, "shots": SHOTS,
        "timestamp": datetime.datetime.utcnow().isoformat()+"Z",
        "theta_msg": THETA_MSG, "varphi_msg": VARPHI_MSG,
        "used_qubits": USED_QUBITS,
        "live_T1T2_fetched": LIVE_AVAILABLE,
    },
    "hardware": {
        "F": hw_F, "F_lo": hw_F_lo, "F_hi": hw_F_hi,
        "p_succ": hw_psucc, "p_lo": hw_p_lo, "p_hi": hw_p_hi,
    },
    "calibration": {
        "readout_errors": {str(k): v for k,v in READOUT_ERRORS.items()},
        "cz_errors"     : {f"{k[0]}-{k[1]}": v for k,v in CZ_ERRORS.items()},
        "T1_us"         : {str(k): v*1e6 for k,v in T1.items()},
        "T2_us"         : {str(k): v*1e6 for k,v in T2.items()},
    },
    "sim_results": sim_results,
    "best_model" : best_key,
}

with open("noisy_sim_validation_results.json","w") as f:
    json.dump(payload, f, indent=2, cls=_Enc)
print("\n[✓] Saved to noisy_sim_validation_results.json")
print("[✓] Run plot_noisy_sim_validation.py for figures.")

  Noisy-Simulator Validation — ibm_torino

  Hardware reference (ibm_torino, 10,000 shots/basis):
    F_msg  = 0.8577  [0.8411, 0.8749]
    p_succ = 0.2430  [0.2346, 0.2515]

  Fetching T1/T2 from ibm_torino ...


qiskit_runtime_service._discover_account:WARNING:2026-03-22 12:54:04,671: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-03-22 12:54:08,233: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-22 12:54:08,235: Using instance: CTCs, plan: open


  [✓] T1/T2 fetched for qubits [45, 46, 55, 65, 66, 67, 68]
    q[45]: T1=43.3µs  T2=53.7µs
    q[46]: T1=110.3µs  T2=103.4µs
    q[55]: T1=169.3µs  T2=101.3µs
    q[65]: T1=163.7µs  T2=145.9µs
    q[66]: T1=266.9µs  T2=226.6µs
    q[67]: T1=240.0µs  T2=20.7µs
    q[68]: T1=173.2µs  T2=71.9µs

  Building noise models ...
  [✓] 5 noise models built

  Running noisy simulations ...

    NM0_noiseless: Noiseless (ideal)
      basis Z: 10000 shots, kept=2458
      basis X: 10000 shots, kept=2464
      basis Y: 10000 shots, kept=2460
      F=0.9972  p_succ=0.2458  ΔF=+0.1395  Δp=+0.0028

    NM1_depol: Depolarising only
      basis Z: 10000 shots, kept=2519
      basis X: 10000 shots, kept=2559
      basis Y: 10000 shots, kept=2440
      F=0.9951  p_succ=0.2519  ΔF=+0.1374  Δp=+0.0089

    NM2_thermal: Depol + thermal (T1/T2)
      basis Z: 10000 shots, kept=2504
      basis X: 10000 shots, kept=2541
      basis Y: 10000 shots, kept=2562
      F=0.9954  p_succ=0.2504  ΔF=+0.1377  Δp=+0.0074